In [ ]:
from pyspark.sql import functions as F

# Helper for weird column names like URXUCR.x
def qcol(col_name):
    return F.col(f"`{col_name}`")

# Load your tables
SCHEMA = "dataattproject"

demographic = spark.table(f"{SCHEMA}.demographic")
diet = spark.table(f"{SCHEMA}.diet")
examination = spark.table(f"{SCHEMA}.examination")
labs = spark.table(f"{SCHEMA}.labs")
medications = spark.table(f"{SCHEMA}.medications")

# Count total rows in labs
labs_total_rows = labs.count()

# Check null count for every lab column except SEQN
lab_cols_to_check = [c for c in labs.columns if c != "SEQN"]

null_count_exprs = [
    F.sum(F.when(qcol(c).isNull(), 1).otherwise(0)).alias(c)
    for c in lab_cols_to_check
]

null_counts_row = labs.agg(*null_count_exprs).collect()[0].asDict()

# Keep columns with <= 40% nulls
kept_lab_cols = ["SEQN"]
dropped_lab_cols = []

for col_name, null_count in null_counts_row.items():
    null_percent = null_count / labs_total_rows

    if null_percent <= 0.40:
        kept_lab_cols.append(col_name)
    else:
        dropped_lab_cols.append((col_name, round(null_percent * 100, 2)))

print("Original lab columns:", len(labs.columns))
print("Kept lab columns:", len(kept_lab_cols))
print("Dropped lab columns:", len(dropped_lab_cols))

print("\nDropped lab columns because they had more than 40% nulls:")
for col_name, null_percent in dropped_lab_cols:
    print(f"{col_name}: {null_percent}% null")

In [ ]:
tables = {
    "demographic": demographic,
    "diet": diet,
    "examination": examination,
    "labs": labs,
    "medications": medications
}

for name, df in tables.items():
    print(name, "has SEQN:", "SEQN" in df.columns)

In [ ]:
labs_filtered = labs.select([qcol(c) for c in kept_lab_cols])

print("Filtered labs rows:", labs_filtered.count())
print("Filtered labs columns:", len(labs_filtered.columns))

display(labs_filtered)

labs_filtered.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA}.labs_filtered_40_null_cutoff"
)

print(f"Saved table: {SCHEMA}.labs_filtered_40_null_cutoff")

In [ ]:
med_agg = (
    medications
    .groupBy("SEQN")
    .agg(
        F.count("*").alias("medication_count"),
        F.concat_ws(", ", F.collect_set(F.col("RXDDRUG").cast("string"))).alias("medications_list"),
        F.concat_ws(", ", F.collect_set(F.col("RXDRSD1").cast("string"))).alias("conditions_list")
    )
)

display(med_agg)

In [ ]:
merged = (
    demographic
    .join(diet, on="SEQN", how="left")
    .join(examination, on="SEQN", how="left")
    .join(labs_filtered, on="SEQN", how="left")
    .join(med_agg, on="SEQN", how="left")
)

merged = merged.fillna({"medication_count": 0})

print("Merged rows:", merged.count())
print("Merged columns:", len(merged.columns))

display(merged)

In [ ]:
merged.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA}.merged_health_panel_filtered_labs"
)

print(f"Saved table: {SCHEMA}.merged_health_panel_filtered_labs")

In [ ]:
%sql
SELECT *
FROM workspace.dataattproject.labs

In [ ]:
def safe_col(col_name, alias_name):
    if col_name in merged.columns:
        return qcol(col_name).alias(alias_name)
    else:
        return F.lit(None).alias(alias_name)

# Main readable columns
base_select_cols = [
    # ID
    F.col("SEQN"),

    # Demographics
    safe_col("RIAGENDR", "gender"),
    safe_col("RIDAGEYR", "age"),
    safe_col("RIDRETH1", "race_ethnicity"),
    safe_col("DMDEDUC2", "education_level"),
    safe_col("INDFMPIR", "income_to_poverty_ratio"),

    # Body / examination
    safe_col("BMXWT", "weight_kg"),
    safe_col("BMXHT", "height_cm"),
    safe_col("BMXBMI", "bmi"),
    safe_col("BMXWAIST", "waist_cm"),

    # Blood pressure readings
    safe_col("BPXSY1", "systolic_bp_1"),
    safe_col("BPXDI1", "diastolic_bp_1"),
    safe_col("BPXSY2", "systolic_bp_2"),
    safe_col("BPXDI2", "diastolic_bp_2"),
    safe_col("BPXSY3", "systolic_bp_3"),
    safe_col("BPXDI3", "diastolic_bp_3"),

    # Diet / nutrition
    safe_col("DR1TKCAL", "calories"),
    safe_col("DR1TPROT", "protein_g"),
    safe_col("DR1TCARB", "carbs_g"),
    safe_col("DR1TSUGR", "sugar_g"),
    safe_col("DR1TFIBE", "fiber_g"),
    safe_col("DR1TTFAT", "total_fat_g"),
    safe_col("DR1TSFAT", "saturated_fat_g"),
    safe_col("DR1TCHOL", "cholesterol_mg"),
    safe_col("DR1TSODI", "sodium_mg"),
    safe_col("DR1TCAFF", "caffeine_mg"),

    # Medications
    safe_col("medication_count", "medication_count"),
    safe_col("medications_list", "medications_list"),
    safe_col("conditions_list", "conditions_list")
]

# Lab columns that survived the 40% null cutoff
# Exclude SEQN because it is already selected
filtered_lab_cols = [c for c in kept_lab_cols if c != "SEQN"]

# Add all kept labs with a lab_ prefix so they are easy to identify
lab_select_cols = [
    qcol(c).alias("lab_" + c.replace(".", "_"))
    for c in filtered_lab_cols
]

health_panel = merged.select(base_select_cols + lab_select_cols)

print("Cleaned health panel rows:", health_panel.count())
print("Cleaned health panel columns:", len(health_panel.columns))

display(health_panel)

In [ ]:
health_panel.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA}.health_panel_all_filtered_labs"
)

print(f"Saved table: {SCHEMA}.health_panel_all_filtered_labs")

In [ ]:
%sql
SELECT *
FROM dataattproject.health_panel_all_filtered_labs
LIMIT 100;

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Start from the cleaned health panel that includes all labs with <= 40% nulls
base_df = spark.table(f"{SCHEMA}.health_panel_all_filtered_labs")

target_rows = 50000
current_rows = base_df.count()
needed_rows = target_rows - current_rows

print("Current real rows:", current_rows)
print("Synthetic rows needed:", needed_rows)

if needed_rows <= 0:
    augmented_50k = base_df.limit(target_rows)
    print("Base table already has at least 50k rows.")
else:
    # Sample existing rows with replacement
    synthetic = (
        base_df
        .sample(
            withReplacement=True,
            fraction=(needed_rows / current_rows) * 1.5,
            seed=42
        )
        .limit(needed_rows)
    )

    # Create new synthetic SEQN values
    w = Window.orderBy(F.monotonically_increasing_id())
    max_seqn = base_df.agg(F.max("SEQN")).collect()[0][0]

    synthetic = (
        synthetic
        .withColumn("row_num", F.row_number().over(w))
        .withColumn("SEQN", F.lit(max_seqn) + F.col("row_num"))
        .drop("row_num")
    )

    print("Synthetic rows created:", synthetic.count())

    # Numeric columns that should NOT get random noise
    exclude_numeric_cols = {
        "SEQN",
        "gender",
        "race_ethnicity",
        "education_level",
        "obesity_flag",
        "high_bp_flag",
        "diabetes_risk_flag",
        "high_cholesterol_flag",
        "risk_score",
        "medication_count"
    }

    # Get numeric columns automatically
    numeric_types = {"int", "bigint", "double", "float", "decimal", "long", "short"}

    numeric_cols = [
        field.name
        for field in base_df.schema.fields
        if field.name not in exclude_numeric_cols
        and any(t in field.dataType.simpleString().lower() for t in numeric_types)
    ]

    print("Numeric columns receiving noise:", len(numeric_cols))

    # Add small random noise to numeric columns
    for col_name in numeric_cols:
        std_val = base_df.select(F.stddev(F.col(f"`{col_name}`")).alias("std")).collect()[0]["std"]

        if std_val is not None and std_val > 0:
            synthetic = synthetic.withColumn(
                col_name,
                F.when(
                    F.col(f"`{col_name}`").isNotNull(),
                    F.col(f"`{col_name}`") + (F.randn(seed=42) * F.lit(0.03 * std_val))
                ).otherwise(F.col(f"`{col_name}`"))
            )

    # Prevent negative values for obvious health/nutrition/lab numeric columns
    for col_name in numeric_cols:
        synthetic = synthetic.withColumn(
            col_name,
            F.when(F.col(f"`{col_name}`") < 0, 0).otherwise(F.col(f"`{col_name}`"))
        )

    # Combine real + synthetic
    augmented_50k = base_df.unionByName(synthetic)

print("Final augmented rows:", augmented_50k.count())
print("Final augmented columns:", len(augmented_50k.columns))

display(augmented_50k)

In [ ]:
# Save final augmented table as a managed Databricks table
augmented_50k.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA}.health_food_augmented_50k"
)

print(f"Saved table: {SCHEMA}.health_food_augmented_50k")

In [ ]:
df = spark.table("dataattproject.health_food_augmented_50k")

print("Rows:", df.count())
print("Columns:", len(df.columns))
display(df.limit(10))

In [ ]:
from pyspark.sql import functions as F

df = spark.table("dataattproject.health_food_augmented_50k")

df_labeled = (
    df
    .withColumn(
        "obesity_flag",
        F.when(F.col("bmi") >= 30, 1).otherwise(0)
    )
    .withColumn(
        "high_bp_flag",
        F.when(
            (F.col("systolic_bp_1") >= 130) | (F.col("diastolic_bp_1") >= 80),
            1
        ).otherwise(0)
    )
    .withColumn(
        "risk_score",
        F.col("obesity_flag") + F.col("high_bp_flag")
    )
    .withColumn(
        "high_health_risk_label",
        F.when(F.col("risk_score") >= 1, 1).otherwise(0)
    )
)

df_labeled.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "dataattproject.health_food_augmented_labeled"
)

print("Saved table: dataattproject.health_food_augmented_labeled")
display(df_labeled.limit(10))

In [ ]:
spark.sql("""
SELECT 
  high_health_risk_label,
  COUNT(*) AS count
FROM dataattproject.health_food_augmented_labeled
GROUP BY high_health_risk_label
ORDER BY high_health_risk_label
""").show()

In [ ]:
from pyspark.sql.types import NumericType

df = spark.table("dataattproject.health_food_augmented_labeled")

label_col = "high_health_risk_label"

exclude_cols = {
    "SEQN",

    # target / derived label columns
    "obesity_flag",
    "high_bp_flag",
    "risk_score",
    "high_health_risk_label",

    # leakage columns used to create the label
    "bmi",
    "systolic_bp_1",
    "diastolic_bp_1",
    "systolic_bp_2",
    "diastolic_bp_2",
    "systolic_bp_3",
    "diastolic_bp_3",
}

feature_cols = [
    field.name
    for field in df.schema.fields
    if isinstance(field.dataType, NumericType)
    and field.name not in exclude_cols
]

print("Number of feature columns:", len(feature_cols))
print(feature_cols)

# sanity check
print("Leakage columns still included?")
print([c for c in exclude_cols if c in feature_cols])

ml_df = df.select(feature_cols + [label_col])
display(ml_df.limit(10))

In [ ]:
from pyspark.ml.feature import Imputer, VectorAssembler
from pyspark.ml import Pipeline

train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)

imputed_cols = [c + "_imputed" for c in feature_cols]

imputer = Imputer(
    inputCols=feature_cols,
    outputCols=imputed_cols
).setStrategy("median")

assembler = VectorAssembler(
    inputCols=imputed_cols,
    outputCol="features",
    handleInvalid="skip"
)

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())

In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

lr = LogisticRegression(
    featuresCol="features",
    labelCol=label_col,
    maxIter=20
)

lr_pipeline = Pipeline(stages=[imputer, assembler, lr])

lr_model = lr_pipeline.fit(train_df)
lr_predictions = lr_model.transform(test_df)

auc_eval = BinaryClassificationEvaluator(
    labelCol=label_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

accuracy_eval = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="accuracy"
)

precision_eval = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_eval = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="weightedRecall"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="f1"
)

lr_auc = auc_eval.evaluate(lr_predictions)
lr_accuracy = accuracy_eval.evaluate(lr_predictions)
lr_precision = precision_eval.evaluate(lr_predictions)
lr_recall = recall_eval.evaluate(lr_predictions)
lr_f1 = f1_eval.evaluate(lr_predictions)

print("Logistic Regression AUC:", lr_auc)
print("Logistic Regression Accuracy:", lr_accuracy)
print("Logistic Regression Precision:", lr_precision)
print("Logistic Regression Recall:", lr_recall)
print("Logistic Regression F1:", lr_f1)

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol=label_col,
    numTrees=100,
    maxDepth=8,
    seed=42
)

rf_pipeline = Pipeline(stages=[imputer, assembler, rf])

rf_model = rf_pipeline.fit(train_df)
rf_predictions = rf_model.transform(test_df)

rf_auc = auc_eval.evaluate(rf_predictions)
rf_accuracy = accuracy_eval.evaluate(rf_predictions)
rf_precision = precision_eval.evaluate(rf_predictions)
rf_recall = recall_eval.evaluate(rf_predictions)
rf_f1 = f1_eval.evaluate(rf_predictions)

print("Random Forest AUC:", rf_auc)
print("Random Forest Accuracy:", rf_accuracy)
print("Random Forest Precision:", rf_precision)
print("Random Forest Recall:", rf_recall)
print("Random Forest F1:", rf_f1)

In [ ]:
ml_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "dataattproject.health_food_ml_ready"
)

print("Saved table: dataattproject.health_food_ml_ready")

In [ ]:
spark.sql("SELECT current_catalog(), current_schema()").show()

In [ ]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.dataattproject.mlflow_volume")
spark.sql("SHOW VOLUMES IN dataattproject").show(truncate=False)

In [ ]:
import mlflow
import mlflow.spark
from mlflow.models.signature import infer_signature

tmp_path = "/Volumes/workspace/dataattproject/mlflow_volume/mlflow_tmp"

# Make sure UC registry is being used
mlflow.set_registry_uri("databricks-uc")

# Use a small raw input example WITHOUT the label column
input_example = train_df.drop(label_col).limit(5).toPandas()

# Run model on the input example to infer output schema
input_spark = spark.createDataFrame(input_example)

prediction_example = (
    rf_model
    .transform(input_spark)
    .select("prediction")
    .toPandas()
)

signature = infer_signature(input_example, prediction_example)

with mlflow.start_run(run_name="health_food_risk_random_forest_pipeline_with_signature"):
    mlflow.log_metric("auc", rf_auc)
    mlflow.log_metric("accuracy", rf_accuracy)
    mlflow.log_metric("precision", rf_precision)
    mlflow.log_metric("recall", rf_recall)
    mlflow.log_metric("f1", rf_f1)

    mlflow.spark.log_model(
        rf_model,
        artifact_path="health_food_risk_pipeline",
        dfs_tmpdir=tmp_path,
        registered_model_name="workspace.dataattproject.health_food_risk_detector",
        signature=signature,
        input_example=input_example
    )

print("Registered model: workspace.dataattproject.health_food_risk_detector")

In [ ]:
import mlflow

model_name = "workspace.dataattproject.health_food_risk_detector"

client = mlflow.MlflowClient(registry_uri="databricks-uc")

versions = client.search_model_versions(f"name = '{model_name}'")

for v in versions:
    print("Model:", v.name)
    print("Version:", v.version)
    print("Status:", v.status)
    print("Run ID:", v.run_id)
    print()

In [ ]:
spark.sql("SHOW TABLES IN dataattproject").show(truncate=False)

In [ ]:
spark.table("dataattproject.health_food_augmented_50k").limit(5).display()

## Databricks Health + Food ML Pipeline Demo

This Databricks pipeline builds an end-to-end machine learning workflow using health, nutrition, lab, and medication data. The goal of the pipeline is to create usable model features and train a model that predicts whether an individual is in a high-health-risk category.

### 1. Data Ingestion and Merging

The pipeline begins by ingesting multiple health-related datasets, including demographic data, dietary/nutrition data, examination data, lab data, and medication data. These datasets are joined using `SEQN`, which acts as the shared person identifier across the tables.

The final merged dataset combines demographic, food, body measurement, blood pressure, lab, and medication-related information into one unified health panel.

### 2. Lab Data Cleaning

The lab dataset contained many sparse columns, so the pipeline removes lab columns with more than 40% missing values before merging them into the final table. This helps prevent extremely incomplete lab features from negatively affecting the model.

The model does still use lab data, but missing lab values are handled with median imputation during training. This means the model can still run if some lab values are missing, but predictions may be less personalized when a user does not have complete lab results.

A future improvement would be to train two versions of the model: one full model that uses lab data and one lightweight model that works without lab data.

### 3. Augmented Dataset

After cleaning and merging the data, the pipeline creates a 50,000-row augmented dataset. This dataset is saved as:

`dataattproject.health_food_augmented_50k`

The augmented dataset is created by sampling from the cleaned health panel and adding small amounts of noise to numeric health and nutrition features. This gives the model more training examples while keeping the generated data realistic.

### 4. Label Creation

The pipeline creates a binary prediction label called:

`high_health_risk_label`

This label represents whether a person is considered high-risk or lower-risk.

- `1` = high health risk
- `0` = lower health risk

The current label is based on obesity and blood pressure risk indicators. The pipeline also creates supporting columns such as:

- `obesity_flag`
- `high_bp_flag`
- `risk_score`

The labeled dataset is saved as:

`dataattproject.health_food_augmented_labeled`

### 5. Feature Engineering and Leakage Prevention

The ML-ready table uses numeric features from demographics, nutrition, medication count, body measurements, and lab values.

To avoid data leakage, columns that were directly used to create the label were removed from the model features. This includes BMI and blood pressure columns. Keeping those columns would make the model unrealistically strong because it would be given the same information used to define the answer.

After removing leakage columns, the final model uses 106 features.

The ML-ready dataset is saved as:

`dataattproject.health_food_ml_ready`

### 6. Model Training

Two Spark ML models were trained and compared:

1. Logistic Regression
2. Random Forest

Both models used a Spark ML pipeline that includes:

- Median imputation for missing numeric values
- Feature vector assembly
- Binary classification model training

### 7. Model Evaluation Results

Logistic Regression achieved:

- AUC: 0.944
- Accuracy: 0.873
- Precision: 0.872
- Recall: 0.873
- F1 Score: 0.872

Random Forest achieved:

- AUC: 0.975
- Accuracy: 0.919
- Precision: 0.918
- Recall: 0.919
- F1 Score: 0.918

Random Forest was selected as the final model because it performed better across all major metrics, especially AUC and F1 score. AUC is important because it measures how well the model separates high-risk and lower-risk individuals across classification thresholds.

### 8. MLflow Model Registration

The final Random Forest pipeline was registered in Unity Catalog using MLflow.

Registered model:

`workspace.dataattproject.health_food_risk_detector`

Model version:

`1`

Model status:

`READY`

This confirms that the model was successfully logged, registered, and is available for future deployment or serving.

### 9. Current Pipeline Status

The current Databricks pipeline successfully completes the following steps:

1. Ingests raw health, nutrition, lab, and medication data
2. Cleans and filters sparse lab columns
3. Merges datasets into a unified health panel
4. Creates a 50,000-row augmented dataset
5. Creates a binary high-health-risk label
6. Removes leakage columns from the model features
7. Builds an ML-ready feature table
8. Trains Logistic Regression and Random Forest models
9. Evaluates both models using AUC, accuracy, precision, recall, and F1
10. Registers the best model in Unity Catalog with MLflow

### 10. Next Steps

The next step would be to expose the registered model through a Databricks model serving endpoint or connect it to the second microservice. That service would take user health and nutrition features as input and return a predicted high-risk or lower-risk classification.